<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 1 · Introducción a Business Analytics en la era de la IA

No hay semana anterior de la que colgarse, así que empezamos por el final: hoy recorres un análisis
completo de Comercial Andina de principio a fin —cargar, describir, graficar, concluir— y ves en qué
celda exacta se rompe. No vienes a aprender pandas; vienes a ver el arco entero para reconocer, en
las quince semanas que siguen, dónde se pierde un análisis. Esta sesión desbloquea las dos cosas que
usarás todo el curso: un cuaderno que corre sin instalar nada y un protocolo para encargarle trabajo
a un asistente de IA sin quedar a merced de lo que te responda.

> **Hoy haces** · Ejecutas un análisis completo sobre la base del curso (90 min): cargas dos tablas,
> describes lo que pasó, produces un gráfico y escribes una conclusión de negocio. Después reproduces
> un informe real con datos correctos y conclusión inválida, y lo corriges con dos líneas de código.
> Cierras abriendo tu bitácora de prompts.
>
> **Entrega** · Este cuaderno ejecutado de arriba a abajo, con los tres ejercicios resueltos y al menos
> tres filas en la bitácora de prompts. Nombre de archivo: `lab_01_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Qué contesta cada peldaño de la analítica

Antes de tocar un archivo, la pregunta es qué tipo de respuesta necesitas. Los cuatro niveles no son
una escalera de dificultad técnica sino de compromiso: cada peldaño obliga a decidir algo más. Saltarse
un peldaño es el origen de la mitad de los proyectos de analítica que se abandonan: nadie predice bien
un negocio que todavía no sabe describir.

In [ ]:
escalera = pd.DataFrame([
    ("Descriptivo",  "¿Qué pasó?",              "Vendimos 2,8 millones en 30 meses",     "Reporte"),
    ("Diagnóstico",  "¿Por qué pasó?",          "Cuenca vende más por cliente que Quito", "Hipótesis"),
    ("Predictivo",   "¿Qué va a pasar?",        "Este cliente dejará de comprar en 90 días", "Prioridad"),
    ("Prescriptivo", "¿Qué hago al respecto?",  "Visitar primero a estos 40 clientes",   "Acción"),
], columns=["nivel", "pregunta", "ejemplo en Comercial Andina", "qué habilita"])
escalera

Hoy te quedas en los dos primeros peldaños. Es deliberado: el descriptivo mal hecho contamina todo lo
que viene después, y es justo donde se rompió el informe que vas a reproducir.

## 2. Cargar: los primeros cinco minutos con una tabla nueva

Comercial Andina es un distribuidor ecuatoriano con tiendas en Quito, Guayaquil, Cuenca, Manta y Loja,
más un canal en línea. Sus datos son los mismos las dieciséis semanas. Empieza por saber qué tamaño
tiene lo que acabas de abrir.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas.csv")
clientes = pd.read_csv(DATOS / "clientes.csv")

print(f"ventas   : {ventas.shape[0]:,} filas × {ventas.shape[1]} columnas")
print(f"clientes : {clientes.shape[0]:,} filas × {clientes.shape[1]} columnas")
ventas.head()

📌 Ochenta mil filas y mil ochocientos clientes. Ojo con lo que representa **una fila**: no es una venta
ni un cliente, es una **línea de factura** (un producto dentro de una factura). Esa distinción parece
un detalle y es el tema completo de la semana 2.

## 3. Describir: qué pasó

Tres números y un rango de fechas. Con eso ya puedes hablar con un gerente: cuánto se facturó, en qué
periodo, cuántas facturas y cuántos clientes hay detrás.

In [ ]:
# La columna fecha viene en dos formatos mezclados; hoy la convertimos y seguimos.
# El porqué de ese "format=mixed" es material de la semana 4.
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)

# El monto de una línea: cantidad × precio, menos el descuento aplicado.
ventas["monto"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento"])

print(f"periodo        : {ventas['fecha'].min():%d-%m-%Y} a {ventas['fecha'].max():%d-%m-%Y}")
print(f"facturación    : {ventas['monto'].sum():,.2f}")
print(f"facturas       : {ventas['factura_id'].nunique():,}")
print(f"clientes       : {ventas['cliente_id'].nunique():,} de {len(clientes):,} en el padrón")
print(f"líneas         : {len(ventas):,}")

Dos millones ochocientos cuarenta y un mil quinientos nueve con cuarenta y ocho, en treinta meses,
repartidos en 17 675 facturas de 1 750 clientes. Fíjate en el último dato: hay 1 800 clientes en el
padrón y solo 1 750 compraron alguna vez. Cincuenta clientes registrados nunca facturaron. Nadie lo
había mirado.

## 4. Un gráfico

Un gráfico bien hecho no ilustra el análisis: **es** el análisis. La regla del curso, desde hoy: el
título lleva la conclusión, no el nombre del eje.

In [ ]:
mensual = ventas.groupby(ventas["fecha"].dt.to_period("M"))["monto"].sum()
mensual.index = mensual.index.to_timestamp()

# El último mes está incompleto: mira lo que pasa si no lo recortas.
print(mensual.tail(3).to_string())

⚠️ Julio de 2026 cierra en **-776,80**. No es una caída del negocio: el archivo se corta el 30 de junio
y en julio solo quedaron notas de crédito de facturas anteriores. Un mes parcial en el borde de la
serie produce siempre una caída falsa. Se recorta y se dice que se recortó.

In [ ]:
serie = mensual[mensual.index < "2026-07-01"]

fig, ax = plt.subplots()
serie.plot(ax=ax, marker="o", linewidth=2)
ax.set_title("Diciembre es el pico del año: 136,7 mil frente a los 94,7 mil del mes promedio")
ax.set_xlabel("")
ax.set_ylabel("facturación mensual")
plt.tight_layout()
plt.show()

print(f"mes más alto : {serie.idxmax():%b-%Y}  {serie.max():,.2f}")
print(f"mes promedio : {serie.mean():,.2f}")
print(f"diciembre-2025 sobre febrero-2025: {serie['2025-12-01'] / serie['2025-02-01']:.2f} veces")

## 5. El informe de apertura: datos correctos, conclusión inválida

Lo que sigue es un informe real, reescrito con los datos de Comercial Andina. Lo firmó un analista
competente, con datos correctos y software correcto. La conclusión es falsa igual.

La pregunta que le hicieron fue: *¿cuál es nuestra mejor ciudad?*

In [ ]:
# Las ciudades vienen escritas de varias formas en clientes.csv. Se unifican aquí;
# el diagnóstico completo de ese problema es la semana 4.
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = ventas.merge(clientes[["cliente_id", "ciudad", "tipo_cliente"]], on="cliente_id", how="left")
por_ciudad = v.groupby("ciudad")["monto"].sum().sort_values(ascending=False)

fig, ax = plt.subplots()
por_ciudad.plot.bar(ax=ax, color="#4C72B0")
ax.set_title("Quito factura 958,7 mil: 1,4 veces Guayaquil y casi cinco veces Loja")
ax.set_xlabel("")
ax.set_ylabel("facturación acumulada")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

total = ventas["monto"].sum()
print(f"cuota de Quito sobre el total : {por_ciudad['Quito'] / total * 100:.1f} %")
print(f"cuota de Loja  sobre el total : {por_ciudad['Loja'] / total * 100:.1f} %")
print(f"Quito / Guayaquil : {por_ciudad['Quito'] / por_ciudad['Guayaquil']:.2f} veces")
print(f"Quito / Loja      : {por_ciudad['Quito'] / por_ciudad['Loja']:.2f} veces")
por_ciudad

Y aquí está la conclusión que llegó al comité, textual:

> «Quito es nuestra mejor plaza: concentra el 33,7 % de la facturación total, 1,38 veces lo de
> Guayaquil y 4,79 veces lo de Loja. Recomendamos reforzar la fuerza comercial de Quito y revisar la
> continuidad de Loja, que apenas aporta el 7,0 %.»

Los números son correctos. Puedes verificarlos tú mismo arriba. La conclusión, en cambio, no se
sostiene: **la pregunta era cuál es la mejor ciudad y lo que se midió fue cuál es la más grande.**

## 6. La corrección: dividir por lo que hace grande a la ciudad

Quito tiene más clientes que Loja. Que facture más no dice nada sobre su desempeño, igual que un curso
de sesenta alumnos no es mejor que uno de veinte por tener más aprobados. La comparación honesta divide
por el tamaño de la base.

In [ ]:
tabla = pd.DataFrame({
    "ventas": por_ciudad,
    "clientes": clientes.groupby("ciudad").size(),
})
tabla["ventas_por_cliente"] = tabla["ventas"] / tabla["clientes"]
tabla = tabla.sort_values("ventas_por_cliente", ascending=False)
tabla

In [ ]:
fig, ax = plt.subplots()
tabla["ventas_por_cliente"].plot.bar(ax=ax, color="#DD8452")
ax.set_title("Cuenca rinde 1 877,62 por cliente: un 32 % más que Quito, la ciudad que llamábamos la mejor")
ax.set_xlabel("")
ax.set_ylabel("facturación por cliente")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

r = tabla.loc["Cuenca", "ventas_por_cliente"] / tabla.loc["Quito", "ventas_por_cliente"]
print(f"Cuenca por cliente : {tabla.loc['Cuenca', 'ventas_por_cliente']:,.2f}")
print(f"Quito por cliente  : {tabla.loc['Quito', 'ventas_por_cliente']:,.2f}")
print(f"Cuenca rinde un {(r - 1) * 100:.1f} % más por cliente")

El orden se da vuelta entero. Quito, que era la primera, baja a la segunda posición y queda a un punto
de Guayaquil. Cuenca, tercera en facturación, es la primera en rendimiento por cliente. Y Loja, la
ciudad que el informe proponía cerrar, rinde 1 282,98 por cliente: un 10 % menos que Quito, no un
quinto, que es lo que sugería el gráfico de barras.

## 7. La decisión que cambia

Un análisis vale lo que vale la decisión que modifica. Pon precio a la diferencia.

In [ ]:
brecha = (tabla.loc["Cuenca", "ventas_por_cliente"] - tabla.loc["Quito", "ventas_por_cliente"])
oportunidad = brecha * tabla.loc["Quito", "clientes"]

print(f"Si cada cliente de Quito rindiera como uno de Cuenca,")
print(f"Quito facturaría {tabla.loc['Quito', 'ventas'] + oportunidad:,.2f} en lugar de {tabla.loc['Quito', 'ventas']:,.2f}")
print(f"Oportunidad no capturada en Quito : {oportunidad:,.2f}")
print(f"Equivale al {oportunidad / ventas['monto'].sum() * 100:.1f} % de la facturación total de la empresa")

📌 El informe original recomendaba **meter más comercial en Quito**. El análisis corregido dice lo
contrario: en Quito ya hay clientes de sobra y lo que falla es cuánto compra cada uno. La acción no es
captar; es entender qué hace Cuenca con su base y replicarlo. Mismo dato, decisión opuesta.

## 8. Protocolo de uso de IA generativa

En este curso el asistente se usa desde hoy y para todo. Lo que se califica no es el código que
devuelve sino **la calidad del encargo y el rigor de la verificación**. El protocolo tiene tres reglas:

1. **Puedes pedir** código, explicaciones, alternativas y crítica de tu propio razonamiento.
2. **Tienes que verificar** todo número que salga del asistente contra una ejecución tuya. Si no puedes
   verificarlo, no puedes usarlo.
3. **Tienes que registrar** el prompt final y una línea sobre qué comprobaste. Eso es la bitácora.

La bitácora se entrega con cada cuaderno. Esta es la plantilla: complétala hoy con lo que le preguntes
al asistente durante la sesión.

In [ ]:
bitacora = pd.DataFrame(columns=[
    "fecha", "objetivo", "prompt_final", "que_devolvio",
    "como_lo_verifique", "veredicto",
])

def registrar(objetivo, prompt_final, que_devolvio, como_lo_verifique, veredicto):
    """Añade una fila a la bitácora. veredicto: 'acepto', 'corrijo' o 'descarto'."""
    global bitacora
    fila = pd.DataFrame([{
        "fecha": pd.Timestamp.today().date(),
        "objetivo": objetivo,
        "prompt_final": prompt_final,
        "que_devolvio": que_devolvio,
        "como_lo_verifique": como_lo_verifique,
        "veredicto": veredicto,
    }])
    bitacora = pd.concat([bitacora, fila], ignore_index=True)
    return bitacora

# Ejemplo resuelto: así se ve una fila bien escrita.
registrar(
    objetivo="Comparar el desempeño de las ciudades sin que el tamaño de la base contamine",
    prompt_final="Tengo un DataFrame de ventas con cliente_id y otro de clientes con ciudad. "
                 "Quiero comparar ciudades de forma justa. Dame dos métricas alternativas a la "
                 "suma de ventas y explica qué sesgo corrige cada una.",
    que_devolvio="Ventas por cliente y ventas por cliente activo en los últimos 90 días",
    como_lo_verifique="Calculé ventas/clientes a mano para Quito y Cuenca y contrasté el orden",
    veredicto="acepto",
)

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: llama a registrar(...) tres veces, una por cada prompt que uses hoy.
# Al menos uno de los tres debe terminar en veredicto="corrijo" o "descarto":
# si aceptas todo lo que te devuelve el asistente, no lo estás verificando.

bitacora

### 🌶️ Ejercicio 1 — Guiado

El informe comparó ciudades. Repite exactamente la misma trampa y la misma corrección con
**`tipo_cliente`**: primero la facturación total de mayoristas frente a minoristas, después la
facturación por cliente de cada grupo. Escribe en una frase qué cambia y qué no.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: v.groupby("tipo_cliente")["monto"].sum()
# Pista 2: el denominador sale de clientes.groupby("tipo_cliente").size()
# Pista 3: aquí, a diferencia de las ciudades, el orden NO se da vuelta. Explica por qué.

### 🔥 Desafío

`ventas_por_cliente` divide por **todos** los clientes del padrón, incluidos los cincuenta que nunca
compraron. Construye una tercera columna, `ventas_por_cliente_activo`, que divida solo entre los
clientes que sí facturaron, y responde: ¿cambia el ranking de ciudades? ¿Cuál de las dos métricas
defenderías ante el gerente comercial y por qué?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: v.dropna(subset=["cliente_id"]).groupby("ciudad")["cliente_id"].nunique()

### 🎯 Reto en clase (15 min)

En parejas. Toma la conclusión del informe («reforzar Quito, revisar la continuidad de Loja») y
escribe **la versión corregida en tres frases**, con una cifra en cada una, lista para un comité que
no va a ver tu código. Después pídele al asistente de IA que la ataque como si fuera el gerente de
Quito defendiendo su presupuesto, y registra el intercambio en la bitácora.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: no necesitas código nuevo, necesitas elegir tres números de los que ya calculaste.
# Escríbelos aquí como print() para que queden en el cuaderno entregado.

## La trampa de hoy

⚠️ **Creer que el problema del análisis es la herramienta.** El informe de apertura se hizo con
software correcto, datos correctos y aritmética correcta. Ninguna versión más nueva de pandas, ningún
modelo más grande, ninguna licencia de Tableau lo habría salvado: el error estaba en la comparación,
no en el cálculo. Los dos números lado a lado.

In [ ]:
comparacion = pd.DataFrame({
    "métrica del informe (total facturado)": por_ciudad,
    "métrica correcta (por cliente)": tabla["ventas_por_cliente"],
})
comparacion["puesto informe"] = comparacion.iloc[:, 0].rank(ascending=False).astype(int)
comparacion["puesto correcto"] = comparacion.iloc[:, 1].rank(ascending=False).astype(int)
comparacion = comparacion.sort_values("puesto informe")

print("Mejor ciudad según el informe   :", comparacion.index[0])
print("Mejor ciudad según la métrica correcta:", comparacion.sort_values("puesto correcto").index[0])
comparacion

Las dos columnas salen del mismo archivo, de la misma máquina y del mismo lenguaje. Lo único que cambia
es el denominador. Si el gerente hubiera preguntado *«¿comparado con qué?»* —la pregunta que abre la
semana 9— el informe no habría salido de la sala.

## Entregable

Sube `lab_01_apellido.ipynb` con:

- El cuaderno ejecutado de arriba a abajo, sin errores y sin celdas salteadas.
- Los tres ejercicios resueltos, cada uno con su celda de código y una frase de interpretación.
- La bitácora con **al menos tres filas propias**, una de ellas con veredicto `corrijo` o `descarto`.
- Al final, en una celda de texto: en qué celda de este cuaderno se toma la decisión que cambia la
  conclusión. Una sola celda, señálala por su contenido.

## Para tu equipo

- Firmen el contrato de grupo y abran el repositorio compartido esta misma semana. La bitácora de
  prompts vive ahí desde hoy, no desde la semana en que se entrega el proyecto.
- Elijan una empresa cuyos datos puedan conseguir de verdad. Un caso brillante sin datos se convierte
  en un ensayo, y este curso no califica ensayos.
- Busquen en su empresa el equivalente del informe de hoy: un indicador que se reporta en total cuando
  debería reportarse por unidad (por cliente, por tienda, por vendedor, por metro cuadrado). Lo van a
  encontrar. Anótenlo: es el punto de partida del proyecto integrador.